In [2]:
import coiled
import fsspec
import s3fs
import numpy as np
import rioxarray
import xarray as xr
import fsspec
import pandas as pd
import logging 
import numpy as np
import pytz
import dask
import re
import requests
import sparse
import time
import warnings
import zarr
from io import BytesIO
from datetime import datetime
from dask.distributed import Client, LocalCluster
from dask.distributed import print
from flox import ReindexArrayType, ReindexStrategy
from flox.xarray import xarray_reduce

import logging
# import pygwalker as pyg

# T0 INSTALL FLOX:
# 1. In home directory (cd ~), downloaded flox tar.gz (because can't install the latest version using conda-forge for some reason): wget https://files.pythonhosted.org/packages/6e/34/6eea00e3f1de745c8adad5a3dafd46c3481294cff8699c20a9b8d80502ed/flox-0.10.4.tar.gz 
# 2. Installed using pip, but it's still putting it in the active Conda environment: pip install /home/dagibbs22/flox-0.10.4.tar.gz

# TO CREATE A NOTEBOOK IN A COILED CLUSTER
# coiled notebook start --region=us-east-1

In [3]:
logging.getLogger("distributed.client").setLevel(logging.ERROR)

In [4]:
# Zarr creation cluster
cluster = coiled.Cluster(
    name="vegetation_zonal_stats",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=50,
    tags={"project": "AFOLU_flux_model"},
    scheduler_vm_types="r7g.xlarge", 
    worker_vm_types="r7g.2xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

╭─────────────────────────────── Coiled Cluster ───────────────────────────────╮
│                   ]8;id=721689;https://cloud.coiled.io/clusters/1276862\https://cloud.coiled.io/clusters/1276862]8;;\                   │
╰──────────────────────────────────────────────────────────────────────────────╯
╭────────────── Overview ──────────────╮╭─────────── Configuration ────────────╮
│                                      ││                                      │
│ Name: vegetation_zonal_stats         ││ Region: us-east-1                    │
│                                      ││                                      │
│ Scheduler Status: started            ││ Scheduler: r7g.xlarge                │
│                                      ││                                      │
│ Dashboard:                           ││ Workers:   r7g.2xlarge (50)          │
│ ]8;id=539929;https://cluster-abbhv.dask.host?token=HYa5Tqv47azqZVLt\https://cluster-abbhv.dask.host?toke]8;;\ ││                                      │
│ ]8;id=539929;https://cluster-abbhv.dask.host?token=HYa5Tqv47azqZVLt\n=HYa5Tqv47azqZVLt]8;;\                   ││ Workers Requested: 50                │
│                                      ││                                      │
╰──────────────────────────────────────╯╰──────────────────────────────────────╯
╭───────────────────────── (2025/11/20 22:38:27 EST) ──────────────────────────╮
│                                                                              │
│                              All workers ready.                              │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

/home/dagibbs22/miniforge3/envs/coiled_20251117/lib/python3.12/site-packages/distributed/client.py:1612: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| lz4     | 4.4.4  | 4.4.5     | 4.4.5   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


In [5]:
def timestr():

    # Define the Eastern Time timezone
    eastern = pytz.timezone('US/Eastern')

    # Get the current time in UTC and convert to Eastern Time
    eastern_time = datetime.now(eastern)

    # Format the time as a string
    return eastern_time.strftime("%Y%m%d_%H_%M_%S")

In [6]:
# Makes xarray dataframe (I think not a dataset) from list of s3 uris.
# This came from Solomon Negusse and I haven't really changed it.
# He said that an online forum suggested using xr.openmfdataset to open non-overlapping geotifs.
def make_xarray_chunks(tile_uris, chunk_size):

    xarray_chunks = xr.open_mfdataset(
        tile_uris.values.tolist(),
        parallel=True,
        chunks={'x': chunk_size, 'y':chunk_size}
    ).squeeze()
    # ).squeeze().persists()  # Need this if reading from geotifs directly, rather then creating zarrs

    return xarray_chunks

In [7]:
# Lists uris in an s3 folder, for creating zarr of them
# per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682201ec-1f84-800a-a9f9-c9564f613208
def list_folder_uris(base_uri):

    # Initializes S3 filesystem
    fs = s3fs.S3FileSystem(anon=False)  # Set anon=True if public bucket
    
    # Lists all files in the directory
    all_files = fs.ls(base_uri)
    
    # Filters for GeoTIFFs
    tif_files = [f"s3://{f}" for f in all_files if f.endswith(".tif")]
    
    # Converts to a Pandas Series
    series = pd.Series(tif_files)
    
    return series

# Extracts file pattern from uri. Assumes that file pattern includes _ha_yr (as it does from the LULUCF model).
def parse_pattern_from_uri(uri_series):

    uri = uri_series.values.tolist()[0]
    # print("Parsing URI:", uri)

    # regex per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/681a538d-55e4-800a-818b-bcf850757ba0
    pattern = r"__([a-zA-Z0-9_]+(?:__?[a-zA-Z0-9_]+)*)_ha_yr_\d{4}.tif$"
    match = re.search(pattern, uri)

    if match:
        return match.group(1)
    else:
        return None

In [8]:
# Crops one input to the other input's extent.
# ref is the reference dataset that is being cropped to. 
# From long chat in https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/684749fe-7b30-800a-ba8b-c502377f2c3a
def safe_crop(ds, ref):
    return ds.sel(x=ref.x, y=ref.y, method="nearest")

In [9]:
# Converts results of flox to coordinate dictionary.
# This code came from Solomon Negusse and I haven't changed it in any substantial way.
def convert_to_coord_dict(flux_results, interval):

    print(f"   Postprocessing {interval}: {timestr()}")
    sparse_data = flux_results.data
    
    dim_names = flux_results.dims
    indices = sparse_data.coords  # tuple of arrays with indices into each dim
    values = sparse_data.data     # non-zero values
    
    coord_dict = {
        dim: flux_results.coords[dim].values[indices[i]]
        for i, dim in enumerate(dim_names)
    }
    coord_dict["value"] = values

    return coord_dict

In [10]:
### Value options for contextual layer values.
### Every contextual layer needs to have all possible values listed here.

# GADM v4.1 adm0 IDs (from Solomon Negusse's notebook)
gadm_adm0_ids = np.array([  0.,   4.,   8.,  10.,  12.,  16.,  20.,  24.,  28.,  31.,  32.,
        36.,  40.,  44.,  48.,  50.,  51.,  52.,  56.,  60.,  64.,  68.,
        70.,  72.,  74.,  76.,  84.,  86.,  90.,  92.,  96., 100., 104.,
       108., 112., 116., 120., 124., 132., 136., 140., 144., 148., 152.,
       156., 158., 162., 166., 170., 174., 175., 178., 180., 184., 188.,
       191., 192., 196., 203., 204., 208., 212., 214., 218., 222., 226.,
       231., 232., 233., 234., 238., 239., 242., 246., 248., 250., 254.,
       258., 260., 262., 266., 268., 270., 275., 276., 288., 292., 296.,
       300., 304., 308., 312., 316., 320., 324., 328., 332., 334., 336.,
       340., 348., 352., 356., 360., 364., 368., 372., 376., 380., 384.,
       388., 392., 398., 400., 404., 408., 410., 414., 417., 418., 422.,
       426., 428., 430., 434., 438., 440., 442., 450., 454., 458., 462.,
       466., 470., 474., 478., 480., 484., 492., 496., 498., 499., 500.,
       504., 508., 512., 516., 520., 524., 528., 531., 533., 534., 535.,
       540., 548., 554., 558., 562., 566.,70., 574., 578., 580., 581.,
       583., 584., 585., 586., 591., 598., 600., 604., 608., 612., 616.,
       620., 624., 626., 630., 634., 638., 642., 643., 646., 652., 654.,
       659., 660., 662., 663., 666., 670., 674., 678., 682., 686., 688.,
       690., 694., 702., 703., 704., 705., 706., 710., 716., 724., 728.,
       729., 732., 740., 744., 748., 752., 756., 760., 762., 764., 768.,
       772., 776., 780., 784., 788., 792., 795., 796., 798., 800., 804.,
       807., 818., 826., 831., 832., 833., 834., 840., 850., 854., 858.,
       860., 862., 876., 882., 887., 894.], dtype=np.uint16)

# Primary forest value options
primary_forest_IFL_codes = np.array([0, 1], dtype=np.uint8)

Code to run zonal stats

In [11]:
# General zonal stat run properties

# Trial 12
model_version = "version_1_0_2_WWF_area"  # model version, from s3 paths that are being read
run_date = "20251120"   # model run date, from s3 paths that are being read
chunk_size = 4000  # pixels

# # Trial 11
# model_version = "version_1_0_2_WWF_area_1_yr_chunks"  # model version, from s3 paths that are being read
# run_date = "20251120"   # model run date, from s3 paths that are being read
# chunk_size = 4000  # pixels

# # Trials 1-10
# model_version = "version_1_0_2_1884_chunks"
# run_date = "20251027"   # model run date, from s3 paths that are being read
# chunk_size = 10000  # pixels

interval_label = '2015_2016'

# s3 folders for model outputs being analyzed
output_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}/"  # Model output path, for inputs to zonal stats

# Analysis layer s3 paths
gross_emis_CO2_folder = f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
gross_emis_non_CO2_folder = f"{output_path}gross_emissions__all_C_pools__non_CO2_only__MgCO2e/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
gross_emis_all_gases_folder = f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
gross_remv_all_pools_folder = f"{output_path}gross_removals__all_C_pools__MgCO2/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
net_flux_all_pools_CO2_folder = f"{output_path}net_flux__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
net_flux_all_pools_all_gases_folder = f"{output_path}net_flux__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
node_folder = f"{output_path}land_state_node/standard_model/annual_intervals/INTERVAL/4000_pixels/{run_date}/"

# Rechunked mega-zarr
model_mega_zarr_s3_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}/mega_zarr/standard_model/annual_intervals/{chunk_size}_pixels/{run_date}/"

# zarrs for layers not from the flux model (only need to created once)
# They are in a central folder, not with their specific geotif tile sets (at least for now-- we could change this)
adm0_folder = "s3://gfw2-data/gadm_administrative_boundaries/v4.1/v4.1.64__from_gfw-data-lake/raster/epsg-4326/10/40000/adm0/gdal-geotiff/" #GADM v4.1
adm0_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/contextual_layer_global_zarr/GADM4_1_adm0_global/20251117/global_GADM41_adm0_20251117.zarr"

pixel_area_folder = "s3://gfw2-data/analyses/umd_area_2013__from_gfw-data-lake/v1.10/raster/epsg-4326/10/40000/area_m/gdal-geotiff/"
pixel_area_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/contextual_layer_global_zarr/pixel_area/20251117/global_pixel_area_20251117.zarr"

primary_forest_IFL_folder = "s3://gfw2-data/climate/carbon_model/ifl_primary_merged/processed/20200724/"
primary_forest_IFL_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/contextual_layer_global_zarr/IFL2000_tropical_primary_forest_2001/20251117/ifl_primary_forest_merged_20251117.zarr"

In [ ]:
# %%time

# # CREATES ZARRS FOR INPUTS NOT GENERATED BY THE AFOLU MODEL
# # THIS SHOULD ONLY EVER HAVE TO BE DONE ONCE FOR EACH INPUT
# # Creating adm0, pixel area and IFL/primary forest used 54 credits $2.95 AWS charges, and 7 minutes (50 r7g.2xlarge workers).
# # https://cloud.coiled.io/clusters/1269448/account/wri-forest-research/information?workspace=WRI-forest-research

# print(f"Reading inputs that apply to all intervals: {timestr()}")

# # GADM adm0
# adm0_uris = list_folder_uris(adm0_folder)
# print("adm0_folder:", adm0_folder)
# print(adm0_uris[0])
# print(f"Tile count in {adm0_folder}: {len(adm0_uris)}")

# print(f"   Reading adm0: {timestr()}")
# adm0_xarray_chunks = make_xarray_chunks(adm0_uris, chunk_size)
# adm0_xarray_chunks['band_data'] = adm0_xarray_chunks['band_data'].astype('uint16')  # adm0 should be uint16 but make_xarray_chunks makes it float64 for some reason
# # print("adm0_xarray_chunks:", adm0_xarray_chunks)  # Print to confirm that the zarr datatype is correct

# print(f"   zarring adm0: {timestr()}")
# adm0_xarray_chunks.to_zarr(adm0_zarr_path, mode='w')
# # remove_FillValue(adm0_zarr_path)  # Added per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6912af84-deb4-832d-81f0-da2b22b0737d to deal with FillValue problems
# print(f"   Finished zarring adm0: {timestr()}")


# # Pixel area
# pixel_area_uris = list_folder_uris(pixel_area_folder)
# print("pixel_area_folder:", pixel_area_folder)
# print(pixel_area_uris[0])
# print(f"Tile count in {pixel_area_folder}: {len(pixel_area_uris)}")

# print(f"   Reading pixel_area: {timestr()}")
# pixel_area_xarray_chunks = make_xarray_chunks(pixel_area_uris, chunk_size)
# print("pixel_area_xarray_chunks:", pixel_area_xarray_chunks)

# print(f"   zarring pixel_area: {timestr()}")
# pixel_area_xarray_chunks.to_zarr(pixel_area_zarr_path, mode='w')
# # remove_FillValue(pixel_area_zarr_path)  # Added per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6912af84-deb4-832d-81f0-da2b22b0737d to deal with FillValue problems
# print(f"   Finished zarring pixel_area: {timestr()}")


# # Humid tropical primary forest/IFL merged
# primary_forest_IFL_uris = list_folder_uris(primary_forest_IFL_folder)
# print("primary_forest_IFL_folder:", primary_forest_IFL_folder)
# print(primary_forest_IFL_uris[0])
# print(f"Tile count in {primary_forest_IFL_folder}: {len(primary_forest_IFL_uris)}")

# print(f"   Reading primary_forest_IFL: {timestr()}")
# primary_forest_IFL_xarray_chunks = make_xarray_chunks(primary_forest_IFL_uris, chunk_size)
# primary_forest_IFL_xarray_chunks['band_data'] = primary_forest_IFL_xarray_chunks['band_data'].astype('uint8')  # should be uint8 but make_xarray_chunks makes it float32 for some reason
# print("primary_forest_IFL_xarray_chunks:", primary_forest_IFL_xarray_chunks)  # Print to confirm that the zarr datatype is correct

# print(f"   zarring primary_forest_IFL: {timestr()}")
# primary_forest_IFL_xarray_chunks.to_zarr(primary_forest_IFL_zarr_path, mode='w')
# # remove_FillValue(primary_forest_IFL_zarr_path)  # Added per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6912af84-deb4-832d-81f0-da2b22b0737d to deal with FillValue problems
# print(f"   Finished zarring primary_forest_IFL: {timestr()}")

In [12]:
# # Get filename patterns from the GeoTIFF URIs if you rely on them later
# gross_emis_CO2_folder_interval = gross_emis_CO2_folder.replace("INTERVAL", interval_label)
# gross_emis_CO2_uris = list_folder_uris(gross_emis_CO2_folder_interval)
# gross_emis_non_CO2_folder_interval = gross_emis_non_CO2_folder.replace("INTERVAL", interval_label)
# gross_emis_non_CO2_uris = list_folder_uris(gross_emis_non_CO2_folder_interval)
# gross_emis_all_gases_folder_interval = gross_emis_all_gases_folder.replace("INTERVAL", interval_label)
# gross_emis_all_gases_uris = list_folder_uris(gross_emis_all_gases_folder_interval)
# gross_remv_all_pools_folder_interval = gross_remv_all_pools_folder.replace("INTERVAL", interval_label)
# gross_remv_all_pools_uris = list_folder_uris(gross_remv_all_pools_folder_interval)
# net_flux_all_pools_CO2_folder_interval = net_flux_all_pools_CO2_folder.replace("INTERVAL", interval_label)
# net_flux_all_pools_CO2_uris = list_folder_uris(net_flux_all_pools_CO2_folder_interval)
# net_flux_all_pools_all_gases_folder_interval = net_flux_all_pools_all_gases_folder.replace("INTERVAL", interval_label)
# net_flux_all_pools_all_gases_uris = list_folder_uris(net_flux_all_pools_all_gases_folder_interval)
# node_folder_interval = node_folder.replace("INTERVAL", interval_label)
# node_tile_year_uris = list_folder_uris(node_folder_interval)
# 
# gross_emis_CO2_output_pattern       = parse_pattern_from_uri(gross_emis_CO2_uris)
# gross_emis_non_CO2_output_pattern   = parse_pattern_from_uri(gross_emis_non_CO2_uris)
# gross_emis_all_gases_output_pattern = parse_pattern_from_uri(gross_emis_all_gases_uris)
# gross_remv_all_pools_output_pattern = parse_pattern_from_uri(gross_remv_all_pools_uris)
# net_flux_CO2_output_pattern         = parse_pattern_from_uri(net_flux_all_pools_CO2_uris)
# net_flux_all_gases_output_pattern   = parse_pattern_from_uri(net_flux_all_pools_all_gases_uris)
# node_output_pattern                 = 'land_state_node'

gross_emis_CO2_output_pattern       = "gross_emissions__all_C_pools__CO2_only__MgCO2"
gross_emis_non_CO2_output_pattern   = "gross_emissions__all_C_pools__non_CO2_only__MgCO2e"
gross_emis_all_gases_output_pattern = "gross_emissions__all_C_pools__all_gases__MgCO2e"
gross_remv_all_pools_output_pattern = "gross_removals__all_C_pools__MgCO2"
net_flux_CO2_output_pattern         = "net_flux__all_C_pools__CO2_only__MgCO2"
net_flux_all_gases_output_pattern   = "net_flux__all_C_pools__all_gases__MgCO2e"
node_output_pattern                 = "land_state_node"

print(gross_emis_CO2_output_pattern)
print(gross_emis_non_CO2_output_pattern)
print(gross_emis_all_gases_output_pattern)
print(gross_remv_all_pools_output_pattern)
print(net_flux_CO2_output_pattern)
print(gross_emis_CO2_output_pattern)
print(net_flux_all_gases_output_pattern)
print(node_output_pattern)

gross_emissions__all_C_pools__CO2_only__MgCO2
gross_emissions__all_C_pools__non_CO2_only__MgCO2e
gross_emissions__all_C_pools__all_gases__MgCO2e
gross_removals__all_C_pools__MgCO2
net_flux__all_C_pools__CO2_only__MgCO2
gross_emissions__all_C_pools__CO2_only__MgCO2
net_flux__all_C_pools__all_gases__MgCO2e
land_state_node


In [13]:
# Opens flux model output mega-zarr
ds_all_global = xr.open_zarr(model_mega_zarr_s3_path)
ds_all_global

<xarray.Dataset> Size: 998TB
Dimensions:                                             (year: 9, y: 720000,
                                                         x: 1440000)
Coordinates:
  * x                                                   (x) float64 12MB -180...
  * y                                                   (y) float64 6MB 90.0 ...
  * year                                                (year) int64 72B 0 ... 8
Data variables: (12/29)
    carbon_density__AGC__MgC                            (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    carbon_density__BGC__MgC                            (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    carbon_density__deadwood_C__MgC                     (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    carbon_density__litter_C__MgC                       (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    carbon_density__non_soil__MgC                       (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    composite_primary_forest                            (year, y, x) uint8 9TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    ...                                                  ...
    net_flux__all_C_pools__all_gases__MgCO2e            (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    net_flux__all_C_pools__CO2_only__MgCO2              (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    net_flux__BGC__MgCO2                                (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    net_flux__deadwood_C__MgCO2                         (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    net_flux__litter_C__MgCO2                           (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    spatial_ref                                         int32 4B ...

In [ ]:
ds_all_global[gross_emis_CO2_output_pattern]

In [14]:
ds_all_global.chunksizes

Frozen({'year': (9,), 'y': (4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 

In [15]:
# Opens non-model output zarrs
pixel_area_xr = xr.open_zarr(pixel_area_zarr_path)
adm0_xr = xr.open_zarr(adm0_zarr_path).rename_vars(band_data='adm0')
pixel_area_xr = xr.open_zarr(pixel_area_zarr_path).rename_vars(band_data='pixel_area')
primary_forest_IFL_xr = xr.open_zarr(primary_forest_IFL_zarr_path).rename_vars(band_data='primary_forest_IFL')
# print(adm0_xr)
pixel_area_xr
# print(primary_forest_IFL_xr)

<xarray.Dataset> Size: 3TB
Dimensions:      (y: 560000, x: 1440000)
Coordinates:
  * x            (x) float64 12MB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
    band         int64 8B ...
  * y            (y) float64 4MB 80.0 80.0 80.0 80.0 ... -60.0 -60.0 -60.0 -60.0
Data variables:
    pixel_area   (y, x) float32 3TB dask.array<chunksize=(10000, 10000), meta=np.ndarray>
    spatial_ref  int64 8B ...

In [ ]:
# # To confirm that the notebook runs things in a Coiled cluster
# pixel_area_xr.sum().compute()

In [ ]:
ds_interval = ds_all_global.sel(year=0)
ds_interval

In [ ]:
# Opens model output datasets for the given interval
gross_emis_CO2               = ds_interval[gross_emis_CO2_output_pattern]
gross_emis_non_CO2           = ds_interval[gross_emis_non_CO2_output_pattern]
gross_emis_all_gases         = ds_interval[gross_emis_all_gases_output_pattern]
gross_remv_all_pools         = ds_interval[gross_remv_all_pools_output_pattern]
net_flux_all_pools_CO2       = ds_interval[net_flux_CO2_output_pattern]
net_flux_all_pools_all_gases = ds_interval[net_flux_all_gases_output_pattern]
gross_emis_CO2
# # State nodes as the reference grid
# state_nodes = ds_interval["land_state_node"].astype("uint32")
# state_nodes.name = "state_nodes"

In [ ]:
reference = pixel_area_xr
pixel_area_aligned          = reference
adm0_aligned                = safe_crop(adm0_xr, reference)
primary_forest_IFL_aligned  = safe_crop(primary_forest_IFL_xr, reference)

gross_emis_CO2_aligned               = safe_crop(gross_emis_CO2, reference)
gross_emis_non_CO2_aligned           = safe_crop(gross_emis_non_CO2, reference)
gross_emis_all_gases_aligned         = safe_crop(gross_emis_all_gases, reference)
gross_remv_all_pools_aligned         = safe_crop(gross_remv_all_pools, reference)
net_flux_all_pools_CO2_aligned       = safe_crop(net_flux_all_pools_CO2, reference)
net_flux_all_pools_all_gases_aligned = safe_crop(net_flux_all_pools_all_gases, reference)
gross_emis_CO2_aligned

In [ ]:
# Pixel area in hectares
pixel_area__ha = (pixel_area_aligned / 10000)['pixel_area'].astype("float32")
pixel_area__ha

In [ ]:
%%time
# Trial 11: global run on model output of 16 chunks, use chunks of 1x4000x4000 and run one year at a time

print("Cropping")
reference = pixel_area_xr
pixel_area_aligned          = reference
adm0_aligned                = safe_crop(adm0_xr, reference)
primary_forest_IFL_aligned  = safe_crop(primary_forest_IFL_xr, reference)
gross_emis_CO2_aligned               = safe_crop(gross_emis_CO2, reference)
gross_emis_non_CO2_aligned           = safe_crop(gross_emis_non_CO2, reference)
gross_emis_all_gases_aligned         = safe_crop(gross_emis_all_gases, reference)
gross_remv_all_pools_aligned         = safe_crop(gross_remv_all_pools, reference)
net_flux_all_pools_CO2_aligned       = safe_crop(net_flux_all_pools_CO2, reference)
net_flux_all_pools_all_gases_aligned = safe_crop(net_flux_all_pools_all_gases, reference)

print("Creating flux cube")
flux_cube = xr.DataArray(
    dask.array.stack([
        (gross_emis_CO2_aligned.data               * pixel_area__ha.data).astype("float32"),
        (gross_emis_all_gases_aligned.data         * pixel_area__ha.data).astype("float32"),
        (gross_remv_all_pools_aligned.data         * pixel_area__ha.data).astype("float32"),
        (net_flux_all_pools_CO2_aligned.data         * pixel_area__ha.data).astype("float32"),
        (pixel_area__ha.data).astype("float32"),
    ]),
    dims=("analysis_layer", "y", "x"),
)

# Final alignment 
print("Aligning")
flux_cube, pixel_area_aligned, adm0_aligned, primary_forest_IFL_aligned = xr.align(
    flux_cube, pixel_area_aligned, adm0_aligned["adm0"], primary_forest_IFL_aligned["primary_forest_IFL"], join="override"
)

print("Computing")
results = xarray_reduce(
    flux_cube,
    *(adm0_aligned, primary_forest_IFL_aligned),
    func='sum',
    expected_groups=(gadm_adm0_ids, primary_forest_IFL_codes, ),
    reindex=ReindexStrategy(blockwise=False, array_type=ReindexArrayType.SPARSE_COO),
    fill_value=0
).compute()

In [26]:
selected_vars = ["gross_emissions__all_C_pools__CO2_only__MgCO2",
                 # "gross_emissions__all_C_pools__non_CO2_only__MgCO2e",
                 # "gross_emissions__all_C_pools__all_gases__MgCO2e",
                 # "gross_removals__all_C_pools__MgCO2",
                 # "net_flux__all_C_pools__CO2_only__MgCO2",
                 # "land_state_node"
                ]

ds_all_global_selected_vars = ds_all_global[selected_vars]
ds_all_global_selected_vars

<xarray.Dataset> Size: 37TB
Dimensions:                                        (year: 9, y: 720000,
                                                    x: 1440000)
Coordinates:
  * x                                              (x) float64 12MB -180.0 .....
  * y                                              (y) float64 6MB 90.0 ... -...
  * year                                           (year) int64 72B 0 1 ... 7 8
Data variables:
    gross_emissions__all_C_pools__CO2_only__MgCO2  (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>

In [27]:
%%time
# Trial 12: use chunks of 9x4000x4000 and analyze all years at the same time

print("Rounding coordinates")
# Fix floating-point precision issues
def round_coords(ds, decimals=5):
    ds = ds.assign_coords({
        'x': np.round(ds.coords['x'].values, decimals),
        'y': np.round(ds.coords['y'].values, decimals)
    })
    return ds

reference = round_coords(pixel_area_xr["pixel_area"])
ds_all_global_selected_vars = round_coords(ds_all_global_selected_vars)
adm0_xr = round_coords(adm0_xr)
primary_forest_IFL_xr = round_coords(primary_forest_IFL_xr)

print("Cropping")
pixel_area_aligned          = reference
adm0_aligned                = safe_crop(adm0_xr, reference)
primary_forest_IFL_aligned  = safe_crop(primary_forest_IFL_xr, reference)
ds_all_global_selected_vars_aligned      = safe_crop(ds_all_global_selected_vars, reference)

print("Creating flux cube")
# List of selected variable names (already aligned and cropped)
selected_datasets = list(ds_all_global_selected_vars_aligned.data_vars)

# Expand pixel_area to match shape of flux variables
pixel_area_expanded = pixel_area_aligned.expand_dims(year=ds_all_global_selected_vars_aligned.year)


# Use the exact same x/y coordinates for both
x_coords = reference.coords['x']
y_coords = reference.coords['y']

# Replace coords in both sources
# pixel_area_expanded = pixel_area_expanded.assign_coords(x=x_coords, y=y_coords)
ds_all_global_selected_vars_aligned = ds_all_global_selected_vars_aligned.assign_coords(x=x_coords, y=y_coords)


# Multiply each flux var by pixel_area
flux_layers = []
for var in selected_datasets:
    flux_scaled = ((ds_all_global_selected_vars_aligned[var] * pixel_area_expanded) / 10000).astype("float32")
    flux_layers.append(flux_scaled)

# Stack into one flux cube: shape (analysis_layer, year, y, x)
flux_cube = xr.concat(flux_layers, dim="analysis_layer")

# Set the analysis_layer coordinate names
flux_cube = flux_cube.assign_coords(
    analysis_layer=("analysis_layer", selected_datasets)
)
flux_cube = round_coords(flux_cube)

# Define bounding box
west, south, east, north = -64, -22, -60, -18

# Subset the flux cube by x/y coordinates
flux_cube_subset = flux_cube.sel(
    x=slice(west, east),
    y=slice(north, south)  # Note: y typically decreases from top to bottom
)
adm0_aligned_subset = adm0_aligned.sel(x=slice(west, east), y=slice(north, south))
primary_forest_IFL_aligned_subset = primary_forest_IFL_aligned.sel(x=slice(west, east), y=slice(north, south))
pixel_area_expanded_subset = pixel_area_expanded.sel(x=slice(west, east), y=slice(north, south))


# print("Flux cube x range:", flux_cube.coords['x'].values.min(), flux_cube.coords['x'].values.max(), "len:", len(flux_cube.coords['x']))
# print("Pixel area x range:", pixel_area_expanded.coords['x'].values.min(), pixel_area_expanded.coords['x'].values.max(), "len:", len(pixel_area_expanded.coords['x']))
# print("ADM0 x range:", adm0_aligned["adm0"].coords['x'].values.min(), adm0_aligned["adm0"].coords['x'].values.max(), "len:", len(adm0_aligned["adm0"].coords['x']))
# print("IFL x range:", primary_forest_IFL_aligned["primary_forest_IFL"].coords['x'].values.min(), primary_forest_IFL_aligned["primary_forest_IFL"].coords['x'].values.max(), "len:", len(primary_forest_IFL_aligned["primary_forest_IFL"].coords['x']))

print("Flux cube x range:", flux_cube_subset.coords['x'].values.min(), flux_cube_subset.coords['x'].values.max(), "len:", len(flux_cube_subset.coords['x']))
print("Pixel area x range:", pixel_area_expanded_subset.coords['x'].values.min(), pixel_area_expanded_subset.coords['x'].values.max(), "len:", len(pixel_area_expanded_subset.coords['x']))
print("ADM0 x range:", adm0_aligned_subset["adm0"].coords['x'].values.min(), adm0_aligned_subset["adm0"].coords['x'].values.max(), "len:", len(adm0_aligned_subset["adm0"].coords['x']))
print("IFL x range:", primary_forest_IFL_aligned_subset["primary_forest_IFL"].coords['x'].values.min(), primary_forest_IFL_aligned_subset["primary_forest_IFL"].coords['x'].values.max(), "len:", len(primary_forest_IFL_aligned_subset["primary_forest_IFL"].coords['x']))

# cube_mean = flux_cube_subset.mean().compute()
# print(cube_mean)

# Final alignment 
print("Aligning")
# flux_cube, pixel_area_expanded, adm0_aligned, primary_forest_IFL_aligned = xr.align(
#     flux_cube, pixel_area_expanded, adm0_aligned["adm0"], primary_forest_IFL_aligned["primary_forest_IFL"], join="override"
# )
flux_cube_subset, pixel_area_expanded_subset, adm0_aligned_subset, primary_forest_IFL_aligned_subset = xr.align(
    flux_cube_subset, pixel_area_expanded_subset, adm0_aligned_subset["adm0"], primary_forest_IFL_aligned_subset["primary_forest_IFL"], join="override"
)

print("Computing")
results = xarray_reduce(
    # flux_cube,
    flux_cube_subset,
    # *(adm0_aligned, primary_forest_IFL_aligned),
    *(adm0_aligned_subset, primary_forest_IFL_aligned_subset),
    func='sum',
    expected_groups=(gadm_adm0_ids, primary_forest_IFL_codes, ),
    group_dims=["year"],
    reindex=ReindexStrategy(blockwise=False, array_type=ReindexArrayType.SPARSE_COO),
    fill_value=0
).compute()

Rounding coordinates
Cropping
Creating flux cube
Flux cube x range: -63.99988 -60.00012 len: 16000
Pixel area x range: -63.99988 -60.00012 len: 16000
ADM0 x range: -63.99988 -60.00012 len: 16000
IFL x range: -63.99988 -60.00012 len: 16000
Aligning
Computing
CPU times: user 4.82 s, sys: 58 ms, total: 4.88 s
Wall time: 13.1 s


In [25]:
flux_cube

<xarray.DataArray (analysis_layer: 2, year: 9, y: 560000, x: 1440000)> Size: 58TB
dask.array<concatenate, shape=(2, 9, 560000, 1440000), dtype=float32, chunksize=(1, 9, 4000, 4000), chunktype=numpy.ndarray>
Coordinates:
  * year            (year) int64 72B 0 1 2 3 4 5 6 7 8
    band            int64 8B 1
  * analysis_layer  (analysis_layer) <U47 376B 'gross_emissions__all_C_pools_...
  * x               (x) float64 12MB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
  * y               (y) float64 4MB 80.0 80.0 80.0 80.0 ... -60.0 -60.0 -60.0

In [23]:
coord_dict = convert_to_coord_dict(results, '2016')
coord_dict

   Postprocessing 2016: 20251120_22_43_45


{'analysis_layer': array(['gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gross_emissions__all_C_pools__CO2_only__MgCO2',
        'gros

In [24]:
df = pd.DataFrame(coord_dict)
df

,analysis_layer,year,adm0,primary_forest_IFL,value
0,gross_emissions__all_C_pools__CO2_only__MgCO2,0,32,1,5.237054e+01
1,gross_emissions__all_C_pools__CO2_only__MgCO2,0,68,0,5.722432e+06
2,gross_emissions__all_C_pools__CO2_only__MgCO2,0,68,1,1.805475e+06
3,gross_emissions__all_C_pools__CO2_only__MgCO2,0,600,0,5.955230e+06
4,gross_emissions__all_C_pools__CO2_only__MgCO2,1,68,0,5.471670e+06
...,...,...,...,...,...
65,gross_emissions__all_C_pools__all_gases__MgCO2e,7,600,0,7.296934e+06
66,gross_emissions__all_C_pools__all_gases__MgCO2e,8,32,1,1.926610e+02
67,gross_emissions__all_C_pools__all_gases__MgCO2e,8,68,0,5.767402e+06
68,gross_emissions__all_C_pools__all_gases__MgCO2e,8,68,1,1.692155e+06


In [ ]:
try:
    client.shutdown()
except Exception:
    pass